# Microsoft GraphRAG（微软研究版）：用“知识图 + 社区摘要”做全局/局部检索

这一节学习的是 **Microsoft Research 的 GraphRAG**（`microsoft.github.io/graphrag`），它和上一节我们手写的“图遍历式 GraphRAG”不太一样：

- **上一节（Graph RAG with LangChain）**：节点=chunk，边=相似度/共享概念，然后在图上做“优先队列遍历”补上下文
- **这一节（Microsoft GraphRAG）**：先用 LLM 从语料里抽取 **实体/关系** → 构建知识图 → 做 **社区发现（community detection）** → 为每个社区生成 **摘要**
  - 查询时提供两种模式：
    - **Local search**：更偏“围绕具体实体/概念”展开（邻域扩展）
    - **Global search**：更偏“全局综合/全局 sensemaking”（利用社区摘要做整体推理）

本 notebook 会沿用原仓库的示例：抓取维基百科的 *Elon Musk* 条目，构建 GraphRAG 索引，并分别演示 `local` 与 `global` 查询。


<div style="text-align: center;">

<img src="../images/Microsoft_GraphRag.svg" alt="Microsoft GraphRAG" style="width:100%; height:auto;">
</div>


## 0) 环境准备

本 notebook **不包含 `pip install`**。运行前请确保安装了：

- `graphrag`（Microsoft GraphRAG）
- `pyyaml`（读写 `settings.yaml`）
- `requests`, `beautifulsoup4`（抓取维基文本）

In [11]:
from __future__ import annotations

import os
import re
import subprocess
import sys
from pathlib import Path

import requests
import yaml
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

# 运行 graphrag CLI 用当前 python，避免环境不一致
PYTHON = sys.executable

In [ ]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

## 1) 获取示例语料：抓取维基百科（Elon Musk）并保存为纯文本

原仓库用的是维基百科的 *Elon Musk* 页面。我们把网页正文转成纯文本并缓存到 `../data/elon.md`（后续重复运行不再请求网页）。


In [13]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ELON_PATH = DATA_DIR / "elon.md"
WIKI_URL = "https://en.wikipedia.org/wiki/Elon_Musk"


def fetch_wikipedia_text(url: str, timeout_s: int = 60) -> str:
    headers = {"User-Agent": "paper-qa-rag-techniques/1.0"}
    r = requests.get(url, headers=headers, timeout=timeout_s)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # 原仓库用 soup.text 并粗略截断；这里沿用同样思路，保证节奏一致
    text = soup.get_text("\n")
    text = text.split("\nSee also")[0]
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


if not ELON_PATH.exists():
    elon_text = fetch_wikipedia_text(WIKI_URL)
    ELON_PATH.write_text(elon_text, encoding="utf-8")
else:
    elon_text = ELON_PATH.read_text(encoding="utf-8")

len(elon_text), str(ELON_PATH)

(8964, '../data/elon.md')

## 2) 初始化 GraphRAG 工程目录（root）

GraphRAG 官方实现提供 CLI：

- `python -m graphrag init --root <root>`：初始化工程结构与 `settings.yaml`
- `python -m graphrag index --root <root>`：执行索引（最耗时、最费 token）

我们把 root 放在：`../data/graphrag_ms`。


In [14]:
GRAPHRAG_ROOT = DATA_DIR / "graphrag_ms"
SETTINGS_PATH = GRAPHRAG_ROOT / "settings.yaml"
INPUT_DIR = GRAPHRAG_ROOT / "input"


def run_cli(args: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    """运行 `python -m graphrag ...`，失败时抛出包含完整输出的异常。"""

    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    p = subprocess.run(
        [PYTHON, "-m", "graphrag", *args],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        capture_output=True,
        text=True,
    )

    out = (p.stdout or "") + ("\n" + p.stderr if p.stderr else "")
    if p.returncode != 0:
        raise RuntimeError(f"graphrag CLI failed (code={p.returncode})\n\n{out}")
    return out


GRAPHRAG_ROOT.mkdir(parents=True, exist_ok=True)

# 初始化时显式指定模型参数，避免进入交互式提示
INIT_MODEL = os.getenv("GRAPHRAG_LLM_MODEL", "gpt-4o")
INIT_EMBEDDING = os.getenv("GRAPHRAG_EMBEDDING_MODEL", "text-embedding-3-large")

if not SETTINGS_PATH.exists():
    # 等价于：python -m graphrag init --root <root> --model ... --embedding ...
    out = run_cli(
        [
            "init",
            "--root",
            str(GRAPHRAG_ROOT),
            "--model",
            INIT_MODEL,
            "--embedding",
            INIT_EMBEDDING,
        ]
    )
    print("\n".join(out.strip().splitlines()[-20:]))

SETTINGS_PATH.exists(), str(GRAPHRAG_ROOT), INIT_MODEL, INIT_EMBEDDING

(True, '../data/graphrag_ms', 'gpt-4o', 'text-embedding-3-large')

## 3) 配置 `settings.yaml`

原仓库 notebook 同时支持 OpenAI / Azure OpenAI。这里保留同样选择：

- `USE_AZURE=0`：用 OpenAI（需要 `OPENAI_API_KEY`）
- `USE_AZURE=1`：用 Azure OpenAI（需要 `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT` 等）

你可以在 `../.env` 里设置这些环境变量。


In [15]:
# 模型与 API Key
LLM_MODEL = os.getenv("GRAPHRAG_LLM_MODEL", "gpt-4o")
EMBEDDING_MODEL = os.getenv("GRAPHRAG_EMBEDDING_MODEL", "text-embedding-3-large")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


def load_settings(path: Path) -> dict:
    return yaml.load(path.read_text(encoding="utf-8"), Loader=yaml.FullLoader)


def dump_settings(path: Path, settings: dict) -> None:
    path.write_text(yaml.dump(settings, sort_keys=False), encoding="utf-8")


def upsert_env_var(env_path: Path, key: str, value: str) -> None:
    lines: list[str] = []
    if env_path.exists():
        lines = env_path.read_text(encoding="utf-8").splitlines()

    found = False
    new_lines: list[str] = []
    for line in lines:
        if line.startswith(f"{key}="):
            new_lines.append(f"{key}={value}")
            found = True
        else:
            new_lines.append(line)

    if not found:
        if new_lines and new_lines[-1].strip() != "":
            new_lines.append("")
        new_lines.append(f"{key}={value}")

    env_path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")


# GraphRAG 会从 <root>/.env 读取 GRAPHRAG_API_KEY（settings.yaml 默认引用 ${GRAPHRAG_API_KEY}）
upsert_env_var(GRAPHRAG_ROOT / ".env", "GRAPHRAG_API_KEY", OPENAI_API_KEY)

settings = load_settings(SETTINGS_PATH)

# 按 settings.yaml 的真实结构写入模型配置
settings["completion_models"]["default_completion_model"]["model_provider"] = "openai"
settings["completion_models"]["default_completion_model"]["model"] = LLM_MODEL
settings["completion_models"]["default_completion_model"]["auth_method"] = "api_key"
settings["completion_models"]["default_completion_model"]["api_key"] = "${GRAPHRAG_API_KEY}"

settings["embedding_models"]["default_embedding_model"]["model_provider"] = "openai"
settings["embedding_models"]["default_embedding_model"]["model"] = EMBEDDING_MODEL
settings["embedding_models"]["default_embedding_model"]["auth_method"] = "api_key"
settings["embedding_models"]["default_embedding_model"]["api_key"] = "${GRAPHRAG_API_KEY}"


dump_settings(SETTINGS_PATH, settings)

{"LLM_MODEL": LLM_MODEL, "EMBEDDING_MODEL": EMBEDDING_MODEL, "env_written": str(GRAPHRAG_ROOT / ".env")}

{'LLM_MODEL': 'gpt-4o',
 'EMBEDDING_MODEL': 'text-embedding-3-large',
 'env_written': '../data/graphrag_ms/.env'}

## 4) 放入 input 文本，并运行 indexing（一次性成本）

GraphRAG 约定把输入文本放在 `<root>/input/*.txt`。

> indexing 会很慢、也会调用很多次 LLM。第一次运行完成后，后续只要不改输入语料，通常不需要重复跑。


In [16]:
INPUT_DIR.mkdir(parents=True, exist_ok=True)

# 原仓库把 elon.md 复制为 elon.txt，这里保持一致
(INPUT_DIR / "elon.txt").write_text(elon_text, encoding="utf-8")

# 等价于：python -m graphrag index --root <root>
# 注意：这一步可能需要较长时间
index_out = run_cli(["index", "--root", str(GRAPHRAG_ROOT)])
print("\n".join(index_out.strip().splitlines()[-30:]))

Workflow complete: create_communities
Starting workflow: create_final_text_units

Workflow complete: create_final_text_units
Starting workflow: create_community_reports
  1 / 2 ..........................................
  2 / 2 ............................................................................................
  1 / 8 ....
  2 / 8 .................
  3 / 8 ..............................
  4 / 8 ..........................................
  5 / 8 ......................................................
  6 / 8 ...................................................................
  7 / 8 ................................................................................
  8 / 8 ............................................................................................
Workflow complete: create_community_reports
Starting workflow: generate_text_embeddings
  1 / 3 .........................
  2 / 3 ...........................................................
  3 / 3 .......................

如果你看到类似“All workflows completed successfully.” 说明索引阶段已完成。


## 5) 查询：Local vs Global

GraphRAG 的查询也通过 CLI：

- `python -m graphrag query --root <root> --method local "<query>"`
- `python -m graphrag query --root <root> --method global "<query>"`

下面用一个 Python 包装函数 `ask_graph()` 来调用 CLI 并清洗输出。


In [17]:
DEFAULT_RESPONSE_TYPE = "用 1-2 段总结，并用要点列出关键信息（最多 300 tokens）"
DEFAULT_MAX_CONTEXT_TOKENS = 10000


def _remove_data_blocks(text: str) -> str:
    # GraphRAG CLI 输出里经常会带类似 [Data: ...] 的附加信息，这里做个简单清洗
    return re.sub(r"\[Data:.*?\]", "", text, flags=re.DOTALL).strip()


def ask_graph(query: str, method: str) -> str:
    if method not in {"local", "global"}:
        raise ValueError("method must be 'local' or 'global'")

    env = {
        # 控制 global search token 上限（不同配置/实现可能会忽略该环境变量）
        "GRAPHRAG_GLOBAL_SEARCH_MAX_TOKENS": str(DEFAULT_MAX_CONTEXT_TOKENS),
    }

    out = run_cli(
        [
            "query",
            "--root",
            str(GRAPHRAG_ROOT),
            "--method",
            method,
            "--response-type",
            DEFAULT_RESPONSE_TYPE,
            query,
        ],
        env=env,
    )

    # 某些输出会带 "Search Response:" 前缀
    if "Search Response:" in out:
        out = out.split("Search Response:", 1)[1]

    return _remove_data_blocks(out)


### 5.1 Local Search：围绕具体实体展开

原仓库示例问的是“马斯克创立了哪些公司/多少家公司”，这类问题更像是围绕实体（Elon Musk）做邻域扩展。


In [18]:
from IPython.display import Markdown

local_query = "What and how many companies and subsidiaries founded by Elon Musk?"
local_result = ask_graph(local_query, "local")
Markdown(local_result)

Elon Musk has founded or co-founded numerous companies and subsidiaries spanning multiple industries, from fintech and transportation to artificial intelligence and space exploration. Below is a summary of the key companies:

---

### **Companies Founded by Elon Musk**

1. **Zip2** (1995, co-founder)  
   - Web software company providing business directories and maps. Sold in 1999 .

2. **X.com** (1999, co-founder)  
   - An online payment platform that later merged with Confinity to form **PayPal** .

3. **SpaceX** (2002, founder and CEO)  
   - Innovates reusable rocket technology and develops space travel and colonization .

4. **Tesla** (2004, co-founder as early investor, CEO since 2008)  
   - A leader in electric vehicles, renewable energy solutions, and energy storage systems .

5. **SolarCity** (2006, co-founder via Tesla)  
   - A solar energy services company later integrated into Tesla as Tesla Energy .

6. **OpenAI** (2015, co-founder)  
   - Focused on advancing artificial intelligence with a commitment to benefiting humanity .

7. **Neuralink** (2016, co-founder)  
   - Develops brain-machine interface technology to connect computers with the human brain .

8. **The Boring Company** (2017, founder)  
   - Specializes in building underground transportation systems, including the **Hyperloop** project .

9. **xAI** (2026, founder and CEO)  
   - A company focused on advancing artificial intelligence as a subsidiary of SpaceX .

10. **Twitter/X (formerly Twitter)** (2022, owner)  
    - Acquired Twitter and later rebranded it as X, leading to significant platform changes .

11. **Musk Foundation** (Established year unspecified, founder)  
    - A philanthropic initiative focusing on clean energy, education, and medical R&D .

---

### **Total Number of Companies Founded or Co-Founded**: **11**

These ventures reflect Musk's ambitious vision to revolutionize industries and address global challenges .

[92m14:57:54 - LiteLLM:WARNING[0m: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
[92m14:57:54 - LiteLLM:WARNING[0m: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'

### 5.2 Global Search：全局综合（sensemaking）

原仓库示例问的是“马斯克的主要成就是什么”，这类问题需要把分散信息做整体综合，更符合 global search 的定位。


In [19]:
global_query = "What are the major accomplishments of Elon Musk?"
global_result = ask_graph(global_query, "global")
Markdown(global_result)

### Summary of Elon Musk's Major Accomplishments

Elon Musk has achieved remarkable milestones across multiple industries, solidifying his position as a leading visionary and entrepreneur.

#### Key Accomplishments:
1. **Space Exploration:**  
   Musk's company SpaceX has revolutionized the aerospace industry through reusable rocket technology and ambitious goals for interplanetary colonization .

2. **Electric Vehicles and Clean Energy:**  
   As the leader of Tesla, Musk has driven innovation in electric vehicles and sustainable energy solutions, transforming global transportation and clean energy industries .

3. **Digital Payments Foundations:**  
   Co-founding X.com, which evolved into PayPal after a merger, Musk contributed to shaping the future of online payments and e-commerce .

4. **Hyperloop Concept:**  
   He proposed the Hyperloop system, an innovative high-speed transportation solution aimed at reducing congestion in urban areas .

5. **Philanthropy:**  
   Through the Musk Foundation, Musk has supported causes such as clean energy adoption, education, space exploration, and medical research .

6. **Artificial Intelligence and Ethics:**  
   Musk founded ventures like xAI and has taken proactive steps to address ethical concerns in artificial intelligence .

7. **Social Media Influence:**  
   Musk’s acquisition and rebranding of Twitter as "X" involved significant operational changes and sparked debates about content moderation .

These achievements underscore his transformative impact on technology, business, and society across diverse disciplines.

[92m14:58:13 - LiteLLM:WARNING[0m: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
[92m14:58:13 - LiteLLM:WARNING[0m: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'

## 6) 你应该如何理解 Local vs Global（对照总结）

- **Local**：更像“从一个实体出发，沿着图的邻居扩展”，适合回答“谁/什么/多少/与 X 相关的事实列表”
- **Global**：更像“先把图分成多个社区，再用社区摘要做全局综合”，适合回答“总体成就/主要主题/全局总结/跨章节整合”

如果你发现：
- local 答得泛：通常是输入文本质量不够、实体抽取/关系抽取质量欠佳，或问题太全局
- global 答得散：通常是社区摘要不够聚合（可能需要更多语料/更合理的摘要设置）
